# Significance tests for `basic_UNet`

This notebook compares the baseline model against selected pruning operating points for the `basic_UNet` experiments.

Important:
- The report tables show aggregated `mean ± std` values.
- Those table values are **not** the data you should test.
- The test should use paired observations from the same evaluation set.

For this project the safest unit is:
- one observation per label volume / case-phase file, created by averaging the slice metrics within that file.

Why:
- your evaluation pipeline computes metrics slice-wise,
- slices from the same volume are correlated,
- using all slices directly would inflate `n` and overstate significance.

Primary analysis:
- metric: foreground mean Dice
- unit: case-volume average over slices
- test: paired Wilcoxon signed-rank test
- effect size: paired mean difference with bootstrap 95% CI

Secondary analysis:
- RV, Myocardium, LV Dice
- optional IoU metrics
- Holm correction for multiple testing


In [1]:
from __future__ import annotations

import json
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml
from scipy.stats import wilcoxon

_cwd = Path.cwd().resolve()
_project_root = next((p for p in (_cwd, *_cwd.parents) if (p / 'src').exists()), None)
if _project_root is None:
    raise RuntimeError('Could not find project root containing src/. Open this notebook from inside basic_UNet.')
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from src.models.unet import UNet
from src.pruning.rebuild import load_full_pruned_model
from src.training.data_loader import SegmentationDataset, _collect_img_lbl_pairs
from src.training.metrics import dice_score, iou_score


## Result-path helpers

These follow the same folder conventions as `metrics_plots.ipynb`:
- baseline: `.../baseline/evaluation/run_summary.json`
- pruned direct eval: `.../pruned/<run_name>/pruned_evaluation/run_summary.json`
- pruned retrained eval: `.../pruned/<run_name>/retrained_pruned_evaluation/run_summary.json`


In [2]:
RESULTS_ROOT = '/mnt/hdd/ttoxopeus/basic_UNet/results'
MODEL_NAME = 'UNet_ACDC'
DEFAULT_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CLASS_NAMES = ['Background', 'RV', 'Myocardium', 'LV']


def _get_pruned_eval_relpath(eval_mode: str) -> str:
    eval_mode = eval_mode.lower().strip()
    if eval_mode in {'retrained', 'retrain', 'ft', 'finetuned'}:
        return os.path.join('retrained_pruned_evaluation', 'run_summary.json')
    if eval_mode in {'pruned', 'direct', 'no_retrain', 'noretrain'}:
        return os.path.join('pruned_evaluation', 'run_summary.json')
    raise ValueError(f'Unknown eval_mode={eval_mode!r}')


def baseline_summary_path(exp_name: str, *, model_name: str = MODEL_NAME, results_root: str = RESULTS_ROOT) -> Path:
    return Path(results_root) / model_name / exp_name / 'baseline' / 'evaluation' / 'run_summary.json'


def pruned_summary_path(exp_name: str, run_name: str, *, eval_mode: str, model_name: str = MODEL_NAME, results_root: str = RESULTS_ROOT) -> Path:
    return Path(results_root) / model_name / exp_name / 'pruned' / run_name / _get_pruned_eval_relpath(eval_mode)


def load_json(path: str | Path) -> dict:
    with open(path, 'r') as f:
        return json.load(f)


def find_experiment_config(start_path: str | Path) -> Path:
    start = Path(start_path).resolve()
    for parent in [start, *start.parents]:
        candidate = parent / 'config.yaml'
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'Could not find config.yaml above {start}')


def unwrap_state_dict(obj):
    if isinstance(obj, dict):
        for k in ['state_dict', 'model_state_dict', 'net', 'model']:
            if k in obj and isinstance(obj[k], dict):
                return obj[k]
    return obj


def extract_case_id(label_path: str | Path) -> str:
    name = Path(label_path).name
    if name.endswith('.nii.gz'):
        return name[:-7]
    return Path(name).stem


def extract_patient_id(case_id: str) -> str:
    match = re.match(r'(.+?)_(ED|ES)$', case_id)
    if match:
        return match.group(1)
    return case_id


## Evaluation helpers

The functions below rebuild the evaluation dataset from the experiment config, load the checkpoint, recompute slice metrics with the project metric definitions, and then aggregate those slice metrics to one row per case-volume.


In [3]:
def load_cfg_for_summary(summary_path: str | Path) -> dict:
    cfg_path = find_experiment_config(summary_path)
    with open(cfg_path, 'r') as f:
        return yaml.safe_load(f)


def load_model_from_eval_summary(summary_path: str | Path, device: str | torch.device = DEFAULT_DEVICE):
    summary = load_json(summary_path)
    cfg = load_cfg_for_summary(summary_path)
    model_cfg = cfg['train']['model']
    ckpt_path = Path(summary['eval']['checkpoint'])
    device = torch.device(device)

    phase = summary['eval']['phase']
    if phase == 'baseline_evaluation':
        model = UNet(
            in_ch=int(model_cfg['in_channels']),
            out_ch=int(model_cfg['out_channels']),
            enc_features=tuple(model_cfg['features']),
        ).to(device)
        state = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(unwrap_state_dict(state))
    else:
        candidate_meta_paths = [
            ckpt_path.with_name(ckpt_path.stem + '_meta.json'),
            ckpt_path.parents[1] / 'pruned_model' / 'pruned_model_meta.json',
        ]
        meta_path = next((p for p in candidate_meta_paths if p.exists()), None)
        if meta_path is None:
            searched = '\n'.join(str(p) for p in candidate_meta_paths)
            raise FileNotFoundError(f'Missing pruning meta file. Searched:\n{searched}')
        meta = load_json(meta_path)
        model = load_full_pruned_model(
            meta=meta,
            ckpt_path=ckpt_path,
            in_ch=int(model_cfg['in_channels']),
            out_ch=int(model_cfg['out_channels']),
            device=device,
        )

    model.eval()
    return model, cfg, summary


def build_eval_dataset_from_summary(summary_path: str | Path):
    cfg = load_cfg_for_summary(summary_path)
    eval_cfg = cfg['evaluation']
    pairs = _collect_img_lbl_pairs(
        eval_cfg['paths']['eval_dir'],
        eval_cfg['paths']['label_dir'],
    )
    dataset = SegmentationDataset(
        img_lbl_pairs=pairs,
        augment=False,
        num_slices_per_volume=eval_cfg.get('num_slices_per_volume'),
    )
    return dataset, cfg


@torch.no_grad()
def compute_slice_metrics(summary_path: str | Path, *, device: str | torch.device = DEFAULT_DEVICE, max_slices: int | None = None, verbose: bool = True) -> pd.DataFrame:
    model, cfg, summary = load_model_from_eval_summary(summary_path, device=device)
    dataset, _ = build_eval_dataset_from_summary(summary_path)
    num_classes = int(cfg['train']['model']['out_channels'])
    device = torch.device(device)

    rows = []
    total = len(dataset) if max_slices is None else min(len(dataset), int(max_slices))
    for idx in range(total):
        img, mask = dataset[idx]
        _img_path, lbl_path, slice_idx = dataset.samples[idx]
        case_id = extract_case_id(lbl_path)

        logits = model(img.unsqueeze(0).to(device))
        mask_b = mask.unsqueeze(0).to(device, dtype=torch.long)
        dice_list = [float(x) for x in dice_score(logits, mask_b, num_classes=num_classes, per_class=True)]
        iou_list = [float(x) for x in iou_score(logits, mask_b, num_classes=num_classes, per_class=True)]

        row = {
            'case_id': case_id,
            'slice_idx': int(slice_idx),
            'foreground_dice': float(np.mean(dice_list[1:])),
            'foreground_iou': float(np.mean(iou_list[1:])),
        }
        for class_idx, class_name in enumerate(CLASS_NAMES):
            slug = class_name.lower().replace(' ', '_')
            row[f'{slug}_dice'] = dice_list[class_idx]
            row[f'{slug}_iou'] = iou_list[class_idx]
        rows.append(row)

    df = pd.DataFrame(rows)
    if verbose:
        print(f'Loaded {summary_path}')
        print(f"phase={summary['eval']['phase']} | slices={len(df)} | case_volumes={df['case_id'].nunique()}")
    return df


def aggregate_to_case_level(slice_df: pd.DataFrame) -> pd.DataFrame:
    metric_cols = [c for c in slice_df.columns if c not in {'case_id', 'slice_idx'}]
    return slice_df.groupby('case_id', as_index=False)[metric_cols].mean().sort_values('case_id').reset_index(drop=True)


def aggregate_to_patient_level(slice_df: pd.DataFrame) -> pd.DataFrame:
    case_df = aggregate_to_case_level(slice_df).copy()
    case_df['patient_id'] = case_df['case_id'].map(extract_patient_id)
    metric_cols = [c for c in case_df.columns if c not in {'case_id', 'patient_id'}]
    patient_df = case_df.groupby('patient_id', as_index=False)[metric_cols].mean()
    return patient_df.rename(columns={'patient_id': 'case_id'}).sort_values('case_id').reset_index(drop=True)


def compare_with_saved_summary(slice_df: pd.DataFrame, summary_path: str | Path) -> pd.Series:
    summary = load_json(summary_path)
    recomputed = {
        'foreground_dice': float(slice_df['foreground_dice'].mean()),
        'foreground_iou': float(slice_df['foreground_iou'].mean()),
        'foreground_dice_std': float(slice_df['foreground_dice'].std(ddof=0)),
        'foreground_iou_std': float(slice_df['foreground_iou'].std(ddof=0)),
    }
    stored = {
        'foreground_dice': float(summary['eval']['foreground']['dice_mean']),
        'foreground_iou': float(summary['eval']['foreground']['iou_mean']),
        'foreground_dice_std': float(summary['eval']['foreground']['dice_std']),
        'foreground_iou_std': float(summary['eval']['foreground']['iou_std']),
    }
    return pd.Series({k: recomputed[k] - stored[k] for k in recomputed}, name='recomputed_minus_saved')


## Significance-test helpers


In [4]:
def align_metric(base_case_df: pd.DataFrame, cand_case_df: pd.DataFrame, metric_col: str) -> pd.DataFrame:
    merged = base_case_df[['case_id', metric_col]].merge(
        cand_case_df[['case_id', metric_col]],
        on='case_id',
        how='inner',
        suffixes=('_baseline', '_candidate'),
    )
    merged = merged.dropna().copy()
    merged['delta'] = merged[f'{metric_col}_candidate'] - merged[f'{metric_col}_baseline']
    return merged


def bootstrap_ci(values: np.ndarray, statistic=np.mean, n_resamples: int = 10000, ci: float = 0.95, seed: int = 42):
    rng = np.random.default_rng(seed)
    values = np.asarray(values, dtype=float)
    idx = rng.integers(0, values.size, size=(n_resamples, values.size))
    boot = np.array([statistic(values[i]) for i in idx])
    alpha = 1.0 - ci
    return tuple(np.quantile(boot, [alpha / 2, 1 - alpha / 2]))


def sign_flip_permutation_pvalue(deltas: np.ndarray, n_resamples: int = 20000, seed: int = 42) -> float:
    rng = np.random.default_rng(seed)
    deltas = np.asarray(deltas, dtype=float)
    observed = abs(deltas.mean())
    signs = rng.choice([-1.0, 1.0], size=(n_resamples, deltas.size))
    sampled = np.abs((signs * deltas).mean(axis=1))
    return float((np.sum(sampled >= observed) + 1) / (n_resamples + 1))


def holm_correction(p_values: list[float]) -> list[float]:
    m = len(p_values)
    order = np.argsort(p_values)
    adjusted = np.empty(m, dtype=float)
    running_max = 0.0
    for rank, idx in enumerate(order):
        candidate = (m - rank) * p_values[idx]
        running_max = max(running_max, candidate)
        adjusted[idx] = min(running_max, 1.0)
    return adjusted.tolist()


def paired_test_report(base_case_df: pd.DataFrame, cand_case_df: pd.DataFrame, metric_col: str, *, seed: int = 42) -> tuple[dict, pd.DataFrame]:
    aligned = align_metric(base_case_df, cand_case_df, metric_col)
    deltas = aligned['delta'].to_numpy(dtype=float)
    if deltas.size == 0:
        raise ValueError(f'No paired case-level observations found for {metric_col}')

    w = wilcoxon(deltas, zero_method='wilcox', alternative='two-sided', correction=False, method='auto')
    mean_ci = bootstrap_ci(deltas, statistic=np.mean, seed=seed)
    report = {
        'metric': metric_col,
        'n_cases': int(deltas.size),
        'baseline_mean': float(aligned[f'{metric_col}_baseline'].mean()),
        'candidate_mean': float(aligned[f'{metric_col}_candidate'].mean()),
        'mean_delta': float(deltas.mean()),
        'median_delta': float(np.median(deltas)),
        'wilcoxon_stat': float(w.statistic),
        'wilcoxon_p': float(w.pvalue),
        'permutation_p': float(sign_flip_permutation_pvalue(deltas, seed=seed)),
        'mean_delta_ci_low': float(mean_ci[0]),
        'mean_delta_ci_high': float(mean_ci[1]),
    }
    return report, aligned


def run_comparison_suite(base_case_df: pd.DataFrame, cand_case_df: pd.DataFrame, *, comparison_name: str, dataset_name: str, primary_metric: str, secondary_metrics: list[str], seed: int = 42):
    primary_report, primary_aligned = paired_test_report(base_case_df, cand_case_df, primary_metric, seed=seed)
    primary_report['dataset'] = dataset_name
    primary_report['comparison'] = comparison_name

    secondary_rows = []
    for metric in secondary_metrics:
        report, _ = paired_test_report(base_case_df, cand_case_df, metric, seed=seed)
        report['dataset'] = dataset_name
        report['comparison'] = comparison_name
        secondary_rows.append(report)

    secondary_df = pd.DataFrame(secondary_rows)
    if not secondary_df.empty:
        secondary_df['holm_p'] = holm_correction(secondary_df['wilcoxon_p'].tolist())

    return pd.DataFrame([primary_report]), secondary_df, primary_aligned


def format_results(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in ['baseline_mean', 'candidate_mean', 'mean_delta', 'median_delta', 'mean_delta_ci_low', 'mean_delta_ci_high']:
        if col in out.columns:
            out[col] = out[col].map(lambda x: f'{x:+.4f}' if 'delta' in col else f'{x:.4f}')
    for col in ['wilcoxon_p', 'permutation_p', 'holm_p']:
        if col in out.columns:
            out[col] = out[col].map(lambda x: f'{x:.4g}')
    if {'mean_delta_ci_low', 'mean_delta_ci_high'}.issubset(out.columns):
        out['mean_delta_95ci'] = out.apply(lambda r: f"[{r['mean_delta_ci_low']}, {r['mean_delta_ci_high']}]", axis=1)
    wanted = [c for c in ['dataset', 'comparison', 'metric', 'n_cases', 'baseline_mean', 'candidate_mean', 'mean_delta', 'median_delta', 'mean_delta_95ci', 'wilcoxon_p', 'permutation_p', 'holm_p'] if c in out.columns]
    return out[wanted]


## Configure the report comparisons

Fill this with the baseline experiment and the two chosen operating points for each dataset table.

Use `eval_mode='retrained'` if the report table uses retrained pruning results.
Use `eval_mode='pruned'` only if you want the direct post-pruning results without retraining.


In [5]:
EXPERIMENTS = {
    'ACDC': {
        'baseline': baseline_summary_path('exp67_uniform_l1_acdc'),
        'operating_points': {
            'best_dice_l1': pruned_summary_path('exp67_uniform_l1_acdc', 'l1_norm_50_50_50_50_50_50_50_50_50_50_50', eval_mode='retrained'),
            'approx_2pct_loss_l1': pruned_summary_path('exp67_uniform_l1_acdc', 'l1_norm_80_80_80_80_80_80_80_80_80_80_80', eval_mode='retrained'),
        },
    },
    'M&M': {
        'baseline': baseline_summary_path('exp74_uniform_l1_MM'),
        'operating_points': {
            'best_dice_l1': pruned_summary_path('exp74_uniform_l1_MM', 'l1_norm_45_45_45_45_45_45_45_45_45_45_45', eval_mode='retrained'),
            'approx_2pct_loss_l1': pruned_summary_path('exp74_uniform_l1_MM', 'l1_norm_85_85_85_85_85_85_85_85_85_85_85', eval_mode='retrained'),
        },
    },
}

PRIMARY_METRIC = 'foreground_dice'
SECONDARY_METRICS = [
    'rv_dice',
    'myocardium_dice',
    'lv_dice',
    'foreground_iou',
    'rv_iou',
    'myocardium_iou',
    'lv_iou',
]

for dataset_name, cfg in EXPERIMENTS.items():
    print(dataset_name)
    print('  baseline:', cfg['baseline'])
    for name, path in cfg['operating_points'].items():
        print(f'  {name}: {path}')


ACDC
  baseline: /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp67_uniform_l1_acdc/baseline/evaluation/run_summary.json
  best_dice_l1: /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp67_uniform_l1_acdc/pruned/l1_norm_50_50_50_50_50_50_50_50_50_50_50/retrained_pruned_evaluation/run_summary.json
  approx_2pct_loss_l1: /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp67_uniform_l1_acdc/pruned/l1_norm_80_80_80_80_80_80_80_80_80_80_80/retrained_pruned_evaluation/run_summary.json
M&M
  baseline: /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp74_uniform_l1_MM/baseline/evaluation/run_summary.json
  best_dice_l1: /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp74_uniform_l1_MM/pruned/l1_norm_45_45_45_45_45_45_45_45_45_45_45/retrained_pruned_evaluation/run_summary.json
  approx_2pct_loss_l1: /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp74_uniform_l1_MM/pruned/l1_norm_85_85_85_85_85_85_85_85_85_85_85/retrained_pruned_evaluation/run_summary.json


## Run the comparisons

The first pass recomputes the slice metrics and then aggregates them to one row per case-volume. This may take some time because it runs the models again.


In [6]:
slice_cache = {}
case_cache = {}
primary_tables = []
secondary_tables = []
aligned_examples = {}

for dataset_name, exp_cfg in EXPERIMENTS.items():
    base_summary = exp_cfg['baseline']
    base_slice_df = compute_slice_metrics(base_summary)
    base_case_df = aggregate_to_patient_level(base_slice_df)
    slice_cache[(dataset_name, 'baseline')] = base_slice_df
    case_cache[(dataset_name, 'baseline')] = base_case_df

    print('\n', dataset_name, 'baseline summary check')
    display(compare_with_saved_summary(base_slice_df, base_summary))

    for comparison_name, cand_summary in exp_cfg['operating_points'].items():
        cand_slice_df = compute_slice_metrics(cand_summary)
        cand_case_df = aggregate_to_patient_level(cand_slice_df)
        slice_cache[(dataset_name, comparison_name)] = cand_slice_df
        case_cache[(dataset_name, comparison_name)] = cand_case_df

        primary_df, secondary_df, aligned_df = run_comparison_suite(
            base_case_df,
            cand_case_df,
            comparison_name=comparison_name,
            dataset_name=dataset_name,
            primary_metric=PRIMARY_METRIC,
            secondary_metrics=SECONDARY_METRICS,
        )
        primary_tables.append(primary_df)
        secondary_tables.append(secondary_df)
        aligned_examples[(dataset_name, comparison_name)] = aligned_df

primary_results = pd.concat(primary_tables, ignore_index=True)
secondary_results = pd.concat(secondary_tables, ignore_index=True)

format_results(primary_results)


Loaded /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp67_uniform_l1_acdc/baseline/evaluation/run_summary.json
phase=baseline_evaluation | slices=1076 | case_volumes=100

 ACDC baseline summary check


foreground_dice       -0.001935
foreground_iou        -0.001720
foreground_dice_std    0.004619
foreground_iou_std     0.004325
Name: recomputed_minus_saved, dtype: float64

✅ Built pruned UNet | enc: [32, 64, 128, 256, 256], dec: [256, 256, 128, 64, 32], bottleneck: 512
Loaded /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp67_uniform_l1_acdc/pruned/l1_norm_50_50_50_50_50_50_50_50_50_50_50/retrained_pruned_evaluation/run_summary.json
phase=retrained_pruned_evaluation | slices=1076 | case_volumes=100
✅ Built pruned UNet | enc: [13, 26, 52, 103, 103], dec: [103, 103, 52, 26, 13], bottleneck: 205
Loaded /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp67_uniform_l1_acdc/pruned/l1_norm_80_80_80_80_80_80_80_80_80_80_80/retrained_pruned_evaluation/run_summary.json
phase=retrained_pruned_evaluation | slices=1076 | case_volumes=100
Loaded /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp74_uniform_l1_MM/baseline/evaluation/run_summary.json
phase=baseline_evaluation | slices=806 | case_volumes=68

 M&M baseline summary check


foreground_dice        0.030185
foreground_iou         0.032377
foreground_dice_std   -0.012066
foreground_iou_std    -0.005900
Name: recomputed_minus_saved, dtype: float64

✅ Built pruned UNet | enc: [36, 71, 141, 282, 282], dec: [282, 282, 141, 71, 36], bottleneck: 564
Loaded /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp74_uniform_l1_MM/pruned/l1_norm_45_45_45_45_45_45_45_45_45_45_45/retrained_pruned_evaluation/run_summary.json
phase=retrained_pruned_evaluation | slices=806 | case_volumes=68
✅ Built pruned UNet | enc: [10, 20, 39, 77, 77], dec: [77, 77, 39, 20, 10], bottleneck: 154
Loaded /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp74_uniform_l1_MM/pruned/l1_norm_85_85_85_85_85_85_85_85_85_85_85/retrained_pruned_evaluation/run_summary.json
phase=retrained_pruned_evaluation | slices=806 | case_volumes=68


,dataset,comparison,metric,n_cases,baseline_mean,candidate_mean,mean_delta,median_delta,mean_delta_95ci,wilcoxon_p,permutation_p
0,ACDC,best_dice_l1,foreground_dice,50,0.8588,0.8690,+0.0102,+0.0072,"[+0.0024, +0.0183]",0.006525,0.01515
1,ACDC,approx_2pct_loss_l1,foreground_dice,50,0.8588,0.8524,-0.0065,-0.0088,"[-0.0149, +0.0022]",0.1543,0.1499
2,M&M,best_dice_l1,foreground_dice,34,0.7870,0.8082,+0.0212,+0.0312,"[+0.0036, +0.0373]",0.02058,0.02145
3,M&M,approx_2pct_loss_l1,foreground_dice,34,0.7870,0.7583,-0.0287,-0.0305,"[-0.0457, -0.0116]",0.006141,0.0033


In [7]:
format_results(secondary_results.sort_values(['dataset', 'comparison', 'metric']).reset_index(drop=True))


,dataset,comparison,metric,n_cases,baseline_mean,candidate_mean,mean_delta,median_delta,mean_delta_95ci,wilcoxon_p,permutation_p,holm_p
0,ACDC,approx_2pct_loss_l1,foreground_iou,50,0.8008,0.7915,-0.0093,-0.0142,"[-0.0178, -0.0007]",0.04294,0.0411,0.1718
1,ACDC,approx_2pct_loss_l1,lv_dice,50,0.8901,0.8885,-0.0017,-0.0030,"[-0.0122, +0.0089]",0.5399,0.764,0.5399
2,ACDC,approx_2pct_loss_l1,lv_iou,50,0.8468,0.8417,-0.0051,-0.0056,"[-0.0155, +0.0054]",0.2297,0.3579,0.4595
3,ACDC,approx_2pct_loss_l1,myocardium_dice,50,0.8394,0.8320,-0.0074,-0.0089,"[-0.0192, +0.0044]",0.0876,0.2329,0.2628
4,ACDC,approx_2pct_loss_l1,myocardium_iou,50,0.7602,0.7489,-0.0113,-0.0119,"[-0.0230, +0.0005]",0.01959,0.0693,0.09794
5,ACDC,approx_2pct_loss_l1,rv_dice,50,0.8470,0.8367,-0.0104,-0.0144,"[-0.0257, +0.0064]",0.0133,0.2161,0.07983
6,ACDC,approx_2pct_loss_l1,rv_iou,50,0.7955,0.7839,-0.0116,-0.0216,"[-0.0270, +0.0055]",0.005406,0.1734,0.03784
7,ACDC,best_dice_l1,foreground_iou,50,0.8008,0.8110,+0.0102,+0.0085,"[+0.0027, +0.0180]",0.008588,0.01245,0.04294
8,ACDC,best_dice_l1,lv_dice,50,0.8901,0.9027,+0.0126,+0.0054,"[+0.0014, +0.0240]",0.005942,0.03185,0.0416
9,ACDC,best_dice_l1,lv_iou,50,0.8468,0.8601,+0.0134,+0.0094,"[+0.0024, +0.0247]",0.005942,0.02245,0.0416


In [8]:
for _, row in primary_results.iterrows():
    print(
        f"{row['dataset']} {row['comparison']}: paired Wilcoxon test on case-level foreground Dice over {int(row['n_cases'])} case-volumes, "
        f"p={row['wilcoxon_p']:.4g}, mean difference={row['mean_delta']:+.4f}, "
        f"95% bootstrap CI=[{row['mean_delta_ci_low']:+.4f}, {row['mean_delta_ci_high']:+.4f}]"
    )


ACDC best_dice_l1: paired Wilcoxon test on case-level foreground Dice over 50 case-volumes, p=0.006525, mean difference=+0.0102, 95% bootstrap CI=[+0.0024, +0.0183]
ACDC approx_2pct_loss_l1: paired Wilcoxon test on case-level foreground Dice over 50 case-volumes, p=0.1543, mean difference=-0.0065, 95% bootstrap CI=[-0.0149, +0.0022]
M&M best_dice_l1: paired Wilcoxon test on case-level foreground Dice over 34 case-volumes, p=0.02058, mean difference=+0.0212, 95% bootstrap CI=[+0.0036, +0.0373]
M&M approx_2pct_loss_l1: paired Wilcoxon test on case-level foreground Dice over 34 case-volumes, p=0.006141, mean difference=-0.0287, 95% bootstrap CI=[-0.0457, -0.0116]


## What to report

Use the primary case-level foreground Dice test in the main report text.

Suggested wording:

> We compared each pruning operating point against the baseline using paired case-level foreground Dice scores. Slice-wise predictions were first aggregated within each label volume, producing one observation per case-volume. Statistical significance was assessed with a two-sided Wilcoxon signed-rank test, and effect sizes were summarized as paired mean Dice differences with bootstrap 95% confidence intervals.

Practical recommendation:
- keep the table values as `mean ± std` for descriptive reporting,
- use the notebook's paired case-level tests for inferential reporting,
- treat class-wise Dice and IoU results as secondary analyses.


## Pearson operating points

Use the cells below if you also want the same paired significance analysis for the Pearson pruning runs. This mirrors the `\ell_1` analysis above, but keeps the outputs separate.


In [9]:
PEARSON_EXPERIMENTS = {
    'ACDC': {
        'baseline': baseline_summary_path('exp67_uniform_l1_acdc'),
        'operating_points': {
            'best_dice_pearson': pruned_summary_path('exp70_uniform_corr_acdc', 'corr_t92_99_99_99_99_99_99_99_99_99_99_99', eval_mode='retrained'),
            'approx_2pct_loss_pearson': pruned_summary_path('exp70_uniform_corr_acdc', 'corr_t80_99_99_99_99_99_99_99_99_99_99_99', eval_mode='retrained'),
        },
    },
    'M&M': {
        'baseline': baseline_summary_path('exp74_uniform_l1_MM'),
        'operating_points': {
            'best_dice_pearson': pruned_summary_path('exp76_uniform_corr_MM', 'corr_t94_99_99_99_99_99_99_99_99_99_99_99', eval_mode='retrained'),
            'approx_2pct_loss_pearson': pruned_summary_path('exp76_uniform_corr_MM', 'corr_t84_99_99_99_99_99_99_99_99_99_99_99', eval_mode='retrained'),
        },
    },
}

for dataset_name, cfg in PEARSON_EXPERIMENTS.items():
    print(dataset_name)
    print('  baseline:', cfg['baseline'])
    for name, path in cfg['operating_points'].items():
        print(f'  {name}: {path}')


ACDC
  baseline: /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp67_uniform_l1_acdc/baseline/evaluation/run_summary.json
  best_dice_pearson: /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp70_uniform_corr_acdc/pruned/corr_t92_99_99_99_99_99_99_99_99_99_99_99/retrained_pruned_evaluation/run_summary.json
  approx_2pct_loss_pearson: /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp70_uniform_corr_acdc/pruned/corr_t80_99_99_99_99_99_99_99_99_99_99_99/retrained_pruned_evaluation/run_summary.json
M&M
  baseline: /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp74_uniform_l1_MM/baseline/evaluation/run_summary.json
  best_dice_pearson: /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp76_uniform_corr_MM/pruned/corr_t94_99_99_99_99_99_99_99_99_99_99_99/retrained_pruned_evaluation/run_summary.json
  approx_2pct_loss_pearson: /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp76_uniform_corr_MM/pruned/corr_t84_99_99_99_99_99_99_99_99_99_99_99/retrained_pruned_evaluation/run_summary.js

In [10]:
pearson_slice_cache = {}
pearson_case_cache = {}
pearson_primary_tables = []
pearson_secondary_tables = []
pearson_aligned_examples = {}

for dataset_name, exp_cfg in PEARSON_EXPERIMENTS.items():
    base_summary = exp_cfg['baseline']
    base_slice_df = compute_slice_metrics(base_summary)
    base_case_df = aggregate_to_patient_level(base_slice_df)
    pearson_slice_cache[(dataset_name, 'baseline')] = base_slice_df
    pearson_case_cache[(dataset_name, 'baseline')] = base_case_df

    print('\n', dataset_name, 'baseline summary check')
    display(compare_with_saved_summary(base_slice_df, base_summary))

    for comparison_name, cand_summary in exp_cfg['operating_points'].items():
        cand_slice_df = compute_slice_metrics(cand_summary)
        cand_case_df = aggregate_to_patient_level(cand_slice_df)
        pearson_slice_cache[(dataset_name, comparison_name)] = cand_slice_df
        pearson_case_cache[(dataset_name, comparison_name)] = cand_case_df

        primary_df, secondary_df, aligned_df = run_comparison_suite(
            base_case_df,
            cand_case_df,
            comparison_name=comparison_name,
            dataset_name=dataset_name,
            primary_metric=PRIMARY_METRIC,
            secondary_metrics=SECONDARY_METRICS,
        )
        pearson_primary_tables.append(primary_df)
        pearson_secondary_tables.append(secondary_df)
        pearson_aligned_examples[(dataset_name, comparison_name)] = aligned_df

pearson_primary_results = pd.concat(pearson_primary_tables, ignore_index=True)
pearson_secondary_results = pd.concat(pearson_secondary_tables, ignore_index=True)

format_results(pearson_primary_results)


Loaded /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp67_uniform_l1_acdc/baseline/evaluation/run_summary.json
phase=baseline_evaluation | slices=1076 | case_volumes=100

 ACDC baseline summary check


foreground_dice       -0.001935
foreground_iou        -0.001720
foreground_dice_std    0.004619
foreground_iou_std     0.004325
Name: recomputed_minus_saved, dtype: float64

✅ Built pruned UNet | enc: [54, 120, 196, 428, 469], dec: [193, 78, 23, 49, 17], bottleneck: 593
Loaded /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp70_uniform_corr_acdc/pruned/corr_t92_99_99_99_99_99_99_99_99_99_99_99/retrained_pruned_evaluation/run_summary.json
phase=retrained_pruned_evaluation | slices=1076 | case_volumes=100
✅ Built pruned UNet | enc: [30, 79, 91, 165, 276], dec: [41, 13, 4, 17, 8], bottleneck: 158
Loaded /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp70_uniform_corr_acdc/pruned/corr_t80_99_99_99_99_99_99_99_99_99_99_99/retrained_pruned_evaluation/run_summary.json
phase=retrained_pruned_evaluation | slices=1076 | case_volumes=100
Loaded /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp74_uniform_l1_MM/baseline/evaluation/run_summary.json
phase=baseline_evaluation | slices=806 | case_volumes=68

 M&M baseline summary check


foreground_dice        0.030185
foreground_iou         0.032377
foreground_dice_std   -0.012066
foreground_iou_std    -0.005900
Name: recomputed_minus_saved, dtype: float64

✅ Built pruned UNet | enc: [58, 123, 215, 463, 484], dec: [259, 107, 29, 59, 24], bottleneck: 738
Loaded /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp76_uniform_corr_MM/pruned/corr_t94_99_99_99_99_99_99_99_99_99_99_99/retrained_pruned_evaluation/run_summary.json
phase=retrained_pruned_evaluation | slices=806 | case_volumes=68
✅ Built pruned UNet | enc: [36, 92, 114, 249, 348], dec: [67, 23, 6, 20, 10], bottleneck: 236
Loaded /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp76_uniform_corr_MM/pruned/corr_t84_99_99_99_99_99_99_99_99_99_99_99/retrained_pruned_evaluation/run_summary.json
phase=retrained_pruned_evaluation | slices=806 | case_volumes=68


,dataset,comparison,metric,n_cases,baseline_mean,candidate_mean,mean_delta,median_delta,mean_delta_95ci,wilcoxon_p,permutation_p
0,ACDC,best_dice_pearson,foreground_dice,50,0.8588,0.8618,+0.0029,-0.0004,"[-0.0045, +0.0103]",0.5086,0.4491
1,ACDC,approx_2pct_loss_pearson,foreground_dice,50,0.8588,0.8514,-0.0074,-0.0085,"[-0.0178, +0.0028]",0.2531,0.1654
2,M&M,best_dice_pearson,foreground_dice,34,0.7870,0.7927,+0.0057,+0.0122,"[-0.0116, +0.0224]",0.4466,0.5281
3,M&M,approx_2pct_loss_pearson,foreground_dice,34,0.7870,0.7964,+0.0094,+0.0194,"[-0.0107, +0.0288]",0.2556,0.3715


In [11]:
format_results(pearson_secondary_results.sort_values(['dataset', 'comparison', 'metric']).reset_index(drop=True))


,dataset,comparison,metric,n_cases,baseline_mean,candidate_mean,mean_delta,median_delta,mean_delta_95ci,wilcoxon_p,permutation_p,holm_p
0,ACDC,approx_2pct_loss_pearson,foreground_iou,50,0.8008,0.7908,-0.0100,-0.0111,"[-0.0205, +0.0003]",0.1032,0.0691,0.5158
1,ACDC,approx_2pct_loss_pearson,lv_dice,50,0.8901,0.8962,+0.0061,+0.0008,"[-0.0047, +0.0169]",0.4843,0.2922,1
2,ACDC,approx_2pct_loss_pearson,lv_iou,50,0.8468,0.8531,+0.0064,+0.0018,"[-0.0040, +0.0171]",0.4376,0.2617,1
3,ACDC,approx_2pct_loss_pearson,myocardium_dice,50,0.8394,0.8399,+0.0006,-0.0039,"[-0.0113, +0.0121]",0.7667,0.9283,1
4,ACDC,approx_2pct_loss_pearson,myocardium_iou,50,0.7602,0.7571,-0.0031,-0.0033,"[-0.0152, +0.0089]",0.5025,0.6154,1
5,ACDC,approx_2pct_loss_pearson,rv_dice,50,0.8470,0.8181,-0.0289,-0.0125,"[-0.0502, -0.0083]",0.006525,0.0098,0.03915
6,ACDC,approx_2pct_loss_pearson,rv_iou,50,0.7955,0.7623,-0.0332,-0.0192,"[-0.0546, -0.0126]",0.001282,0.00345,0.008976
7,ACDC,best_dice_pearson,foreground_iou,50,0.8008,0.8037,+0.0029,+0.0005,"[-0.0045, +0.0103]",0.5399,0.4518,1
8,ACDC,best_dice_pearson,lv_dice,50,0.8901,0.8980,+0.0079,+0.0039,"[-0.0015, +0.0174]",0.04193,0.1137,0.2516
9,ACDC,best_dice_pearson,lv_iou,50,0.8468,0.8565,+0.0097,+0.0062,"[+0.0005, +0.0190]",0.02485,0.0473,0.1739


In [12]:
for _, row in pearson_primary_results.iterrows():
    print(
        f"{row['dataset']} {row['comparison']}: paired Wilcoxon test on case-level foreground Dice over {int(row['n_cases'])} case-volumes, "
        f"p={row['wilcoxon_p']:.4g}, mean difference={row['mean_delta']:+.4f}, "
        f"95% bootstrap CI=[{row['mean_delta_ci_low']:+.4f}, {row['mean_delta_ci_high']:+.4f}]"
    )


ACDC best_dice_pearson: paired Wilcoxon test on case-level foreground Dice over 50 case-volumes, p=0.5086, mean difference=+0.0029, 95% bootstrap CI=[-0.0045, +0.0103]
ACDC approx_2pct_loss_pearson: paired Wilcoxon test on case-level foreground Dice over 50 case-volumes, p=0.2531, mean difference=-0.0074, 95% bootstrap CI=[-0.0178, +0.0028]
M&M best_dice_pearson: paired Wilcoxon test on case-level foreground Dice over 34 case-volumes, p=0.4466, mean difference=+0.0057, 95% bootstrap CI=[-0.0116, +0.0224]
M&M approx_2pct_loss_pearson: paired Wilcoxon test on case-level foreground Dice over 34 case-volumes, p=0.2556, mean difference=+0.0094, 95% bootstrap CI=[-0.0107, +0.0288]


## Direct \(\ell_1\) vs Pearson comparisons

The baseline-only tests above show whether each method differs from the unpruned model. The cells below directly test whether the selected \(\ell_1\) operating points outperform the corresponding Pearson operating points, which is the comparison needed to support a method-vs-method claim.


In [13]:
DIRECT_METHOD_COMPARISONS = {
    'ACDC': {
        'best_dice': {
            'l1': EXPERIMENTS['ACDC']['operating_points']['best_dice_l1'],
            'pearson': PEARSON_EXPERIMENTS['ACDC']['operating_points']['best_dice_pearson'],
        },
        'approx_2pct_loss': {
            'l1': EXPERIMENTS['ACDC']['operating_points']['approx_2pct_loss_l1'],
            'pearson': PEARSON_EXPERIMENTS['ACDC']['operating_points']['approx_2pct_loss_pearson'],
        },
    },
    'M&M': {
        'best_dice': {
            'l1': EXPERIMENTS['M&M']['operating_points']['best_dice_l1'],
            'pearson': PEARSON_EXPERIMENTS['M&M']['operating_points']['best_dice_pearson'],
        },
        'approx_2pct_loss': {
            'l1': EXPERIMENTS['M&M']['operating_points']['approx_2pct_loss_l1'],
            'pearson': PEARSON_EXPERIMENTS['M&M']['operating_points']['approx_2pct_loss_pearson'],
        },
    },
}

for dataset_name, cfg in DIRECT_METHOD_COMPARISONS.items():
    print(dataset_name)
    for op_name, paths in cfg.items():
        print(f'  {op_name}')
        print('    l1     :', paths['l1'])
        print('    pearson:', paths['pearson'])


ACDC
  best_dice
    l1     : /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp67_uniform_l1_acdc/pruned/l1_norm_50_50_50_50_50_50_50_50_50_50_50/retrained_pruned_evaluation/run_summary.json
    pearson: /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp70_uniform_corr_acdc/pruned/corr_t92_99_99_99_99_99_99_99_99_99_99_99/retrained_pruned_evaluation/run_summary.json
  approx_2pct_loss
    l1     : /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp67_uniform_l1_acdc/pruned/l1_norm_80_80_80_80_80_80_80_80_80_80_80/retrained_pruned_evaluation/run_summary.json
    pearson: /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp70_uniform_corr_acdc/pruned/corr_t80_99_99_99_99_99_99_99_99_99_99_99/retrained_pruned_evaluation/run_summary.json
M&M
  best_dice
    l1     : /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp74_uniform_l1_MM/pruned/l1_norm_45_45_45_45_45_45_45_45_45_45_45/retrained_pruned_evaluation/run_summary.json
    pearson: /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp

In [14]:
direct_method_tables = []
direct_method_aligned = {}

for dataset_name, dataset_cfg in DIRECT_METHOD_COMPARISONS.items():
    for operating_point, paths in dataset_cfg.items():
        l1_slice_df = compute_slice_metrics(paths['l1'])
        pearson_slice_df = compute_slice_metrics(paths['pearson'])

        l1_case_df = aggregate_to_patient_level(l1_slice_df)
        pearson_case_df = aggregate_to_patient_level(pearson_slice_df)

        report, aligned_df = paired_test_report(pearson_case_df, l1_case_df, PRIMARY_METRIC)
        report['dataset'] = dataset_name
        report['comparison'] = operating_point
        report['reference'] = 'pearson'
        report['candidate'] = 'l1'
        direct_method_tables.append(report)
        direct_method_aligned[(dataset_name, operating_point)] = aligned_df

direct_method_results = pd.DataFrame(direct_method_tables)
format_results(direct_method_results)


✅ Built pruned UNet | enc: [32, 64, 128, 256, 256], dec: [256, 256, 128, 64, 32], bottleneck: 512
Loaded /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp67_uniform_l1_acdc/pruned/l1_norm_50_50_50_50_50_50_50_50_50_50_50/retrained_pruned_evaluation/run_summary.json
phase=retrained_pruned_evaluation | slices=1076 | case_volumes=100
✅ Built pruned UNet | enc: [54, 120, 196, 428, 469], dec: [193, 78, 23, 49, 17], bottleneck: 593
Loaded /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp70_uniform_corr_acdc/pruned/corr_t92_99_99_99_99_99_99_99_99_99_99_99/retrained_pruned_evaluation/run_summary.json
phase=retrained_pruned_evaluation | slices=1076 | case_volumes=100
✅ Built pruned UNet | enc: [13, 26, 52, 103, 103], dec: [103, 103, 52, 26, 13], bottleneck: 205
Loaded /mnt/hdd/ttoxopeus/basic_UNet/results/UNet_ACDC/exp67_uniform_l1_acdc/pruned/l1_norm_80_80_80_80_80_80_80_80_80_80_80/retrained_pruned_evaluation/run_summary.json
phase=retrained_pruned_evaluation | slices=1076 | case_volumes

,dataset,comparison,metric,n_cases,baseline_mean,candidate_mean,mean_delta,median_delta,mean_delta_95ci,wilcoxon_p,permutation_p
0,ACDC,best_dice,foreground_dice,50,0.8618,0.8690,+0.0073,+0.0082,"[+0.0002, +0.0146]",0.0693,0.0548
1,ACDC,approx_2pct_loss,foreground_dice,50,0.8514,0.8524,+0.0010,+0.0012,"[-0.0080, +0.0101]",0.9695,0.8363
2,M&M,best_dice,foreground_dice,34,0.7927,0.8082,+0.0155,+0.0119,"[-0.0041, +0.0363]",0.228,0.1455
3,M&M,approx_2pct_loss,foreground_dice,34,0.7964,0.7583,-0.0381,-0.0290,"[-0.0549, -0.0210]",0.0001265,0.0003


In [15]:
for _, row in direct_method_results.iterrows():
    print(
        f"{row['dataset']} {row['comparison']}: paired Wilcoxon test comparing l1 vs Pearson on patient-level foreground Dice over {int(row['n_cases'])} patients, "
        f"p={row['wilcoxon_p']:.4g}, mean difference (l1 - Pearson)={row['mean_delta']:+.4f}, "
        f"95% bootstrap CI=[{row['mean_delta_ci_low']:+.4f}, {row['mean_delta_ci_high']:+.4f}]"
    )


ACDC best_dice: paired Wilcoxon test comparing l1 vs Pearson on patient-level foreground Dice over 50 patients, p=0.0693, mean difference (l1 - Pearson)=+0.0073, 95% bootstrap CI=[+0.0002, +0.0146]
ACDC approx_2pct_loss: paired Wilcoxon test comparing l1 vs Pearson on patient-level foreground Dice over 50 patients, p=0.9695, mean difference (l1 - Pearson)=+0.0010, 95% bootstrap CI=[-0.0080, +0.0101]
M&M best_dice: paired Wilcoxon test comparing l1 vs Pearson on patient-level foreground Dice over 34 patients, p=0.228, mean difference (l1 - Pearson)=+0.0155, 95% bootstrap CI=[-0.0041, +0.0363]
M&M approx_2pct_loss: paired Wilcoxon test comparing l1 vs Pearson on patient-level foreground Dice over 34 patients, p=0.0001265, mean difference (l1 - Pearson)=-0.0381, 95% bootstrap CI=[-0.0549, -0.0210]


## Slice-wise direct \(\ell_1\) vs Pearson comparisons

The cells below repeat the direct method-to-method comparison at the slice level. This is more exploratory than the patient-level analysis because slices from the same patient are not independent, but it can still be useful for inspecting the behavior of the two criteria at finer granularity.


In [ ]:
slicewise_direct_tables = []
slicewise_direct_aligned = {}

for dataset_name, dataset_cfg in DIRECT_METHOD_COMPARISONS.items():
    for operating_point, paths in dataset_cfg.items():
        l1_slice_df = compute_slice_metrics(paths['l1'])
        pearson_slice_df = compute_slice_metrics(paths['pearson'])

        l1_case_df = aggregate_to_case_level(l1_slice_df)
        pearson_case_df = aggregate_to_case_level(pearson_slice_df)

        report, aligned_df = paired_test_report(pearson_case_df, l1_case_df, PRIMARY_METRIC)
        report['dataset'] = dataset_name
        report['comparison'] = operating_point
        report['reference'] = 'pearson'
        report['candidate'] = 'l1'
        slicewise_direct_tables.append(report)
        slicewise_direct_aligned[(dataset_name, operating_point)] = aligned_df

slicewise_direct_results = pd.DataFrame(slicewise_direct_tables)
format_results(slicewise_direct_results)


In [ ]:
for _, row in slicewise_direct_results.iterrows():
    print(
        f"{row['dataset']} {row['comparison']}: paired Wilcoxon test comparing l1 vs Pearson on slice-wise foreground Dice over {int(row['n_cases'])} case-volumes, "
        f"p={row['wilcoxon_p']:.4g}, mean difference (l1 - Pearson)={row['mean_delta']:+.4f}, "
        f"95% bootstrap CI=[{row['mean_delta_ci_low']:+.4f}, {row['mean_delta_ci_high']:+.4f}]"
    )
